In [1]:
import json

# List of input JSON files
input_files = [
    "triplets_chem_dis_cleaned.json",
    "triplets_exposure_cleaned.json",
    "triplets_pheno_cleaned.json"
]

# Dictionary to hold loaded data for each file
datasets = {}

for file_path in input_files:
    with open(file_path, 'r') as f:
        data = json.load(f)        # Load JSON list of triplets
    datasets[file_path] = data
    print(f"{file_path}: loaded {len(data)} triplets")


triplets_chem_dis_cleaned.json: loaded 470725 triplets
triplets_exposure_cleaned.json: loaded 5820 triplets
triplets_pheno_cleaned.json: loaded 156625 triplets


In [2]:
def is_nonempty_str(s):
    """Return True if s is a string that is not empty/blank."""
    return isinstance(s, str) and s.strip() != ""

def validate_triplets(triplets):
    """
    Filter a list of triplet dictionaries, returning only those that are valid.
    A valid triplet has non-empty strings for 'head', 'relation', 'tail'.
    Optional fields like 'chemical_id' or 'disease_id' (if present) must be non-empty strings.
    """
    valid_entries = []
    for entry in triplets:
        # Check required fields
        if not (is_nonempty_str(entry.get('head')) and 
                is_nonempty_str(entry.get('relation')) and 
                is_nonempty_str(entry.get('tail'))):
            # Skip this entry if any required field is missing or empty
            continue
        
        # Check optional identifier fields
        if 'chemical_id' in entry and not is_nonempty_str(entry['chemical_id']):
            continue  # Invalid chemical_id, skip entry
        if 'disease_id' in entry and not is_nonempty_str(entry['disease_id']):
            continue  # Invalid disease_id, skip entry
        
        # (Additional optional fields can be checked similarly, if needed)
        
        # If all checks passed, this entry is valid
        valid_entries.append(entry)
    return valid_entries


In [3]:
for file_path, triplet_list in datasets.items():
    original_count = len(triplet_list)
    # Validate and filter the triplets
    valid_triplets = validate_triplets(triplet_list)
    valid_count = len(valid_triplets)
    dropped_count = original_count - valid_count
    
    # Save the valid triplets to a new JSON file (replace "_cleaned" with "_validated" in name)
    output_path = file_path.replace("_cleaned.json", "_validated.json")
    with open(output_path, 'w') as out_f:
        json.dump(valid_triplets, out_f, indent=4)
    
    # Print summary of results for this dataset
    dataset_name = file_path.replace("_cleaned.json", "")
    print(f"{dataset_name}: {valid_count} valid triplets, {dropped_count} dropped")


triplets_chem_dis: 470725 valid triplets, 0 dropped
triplets_exposure: 5820 valid triplets, 0 dropped
triplets_pheno: 156625 valid triplets, 0 dropped


Unifying Biomedical Triplet Datasets


1. Loading the JSON Datasets


In [4]:
import json

def load_and_label(file_path, source_label):
    """Load a JSON file and add a 'source' field to each triplet."""
    print(f"Loading {file_path}...")
    with open(file_path, 'r') as f:
        data = json.load(f)
    print(f"  Loaded {len(data)} triplets from {file_path}")
    # Add source label to each triplet
    for triplet in data:
        triplet['source'] = source_label
    print(f"  Added source='{source_label}' to each triplet\n")
    return data

# Load each dataset with an appropriate source label
chem_dis_triplets = load_and_label('triplets_chem_dis_validated.json', 'chem_dis')
exposure_triplets = load_and_label('triplets_exposure_validated.json', 'exposure')
pheno_triplets    = load_and_label('triplets_pheno_validated.json', 'pheno')


Loading triplets_chem_dis_validated.json...
  Loaded 470725 triplets from triplets_chem_dis_validated.json
  Added source='chem_dis' to each triplet

Loading triplets_exposure_validated.json...
  Loaded 5820 triplets from triplets_exposure_validated.json
  Added source='exposure' to each triplet

Loading triplets_pheno_validated.json...
  Loaded 156625 triplets from triplets_pheno_validated.json
  Added source='pheno' to each triplet



2. Normalizing Triplets and Adding Source Fields


In [5]:
{
    "head": "Aspirin",
    "relation": "treats",
    "tail": "Headache",
    "chemical_id": "CHEBI:15365",
    "disease_id": "DOID:0050334",
    "pubmed_ids": [12345678, 23456789],
    "source": "chem_dis"
}


{'head': 'Aspirin',
 'relation': 'treats',
 'tail': 'Headache',
 'chemical_id': 'CHEBI:15365',
 'disease_id': 'DOID:0050334',
 'pubmed_ids': [12345678, 23456789],
 'source': 'chem_dis'}

3. Combining Triplets from All Sources


In [6]:
# Merge all triplets into a single list
all_triplets = chem_dis_triplets + exposure_triplets + pheno_triplets
total_triplets = len(all_triplets)
print(f"Combined total triplets (before de-duplication): {total_triplets}")


Combined total triplets (before de-duplication): 633170


4. Removing Duplicate Triplets


In [7]:
def remove_duplicates(triplets):
    """Remove exact duplicate dictionaries from a list of triplets."""
    seen = set()
    unique_triplets = []
    for triplet in triplets:
        # Create a JSON string representation with sorted keys as a unique signature
        triplet_signature = json.dumps(triplet, sort_keys=True)
        if triplet_signature not in seen:
            unique_triplets.append(triplet)
            seen.add(triplet_signature)
    return unique_triplets

# Remove duplicates from the combined list
unique_triplets = remove_duplicates(all_triplets)
final_count = len(unique_triplets)
duplicates_removed = total_triplets - final_count
print(f"Duplicates removed: {duplicates_removed}")
print(f"Total unique triplets: {final_count}")


Duplicates removed: 0
Total unique triplets: 633170


5. Saving the Unified Triplets to JSON


In [8]:
output_path = "triplets_unified.json"
print(f"Saving unified triplets to {output_path} ...")
with open(output_path, 'w') as f:
    json.dump(unique_triplets, f, indent=4)
print("Save complete.")


Saving unified triplets to triplets_unified.json ...
Save complete.


6. Save Unified Triplets to CSV

In [9]:
import csv

# Define output CSV file path
csv_output_path = "triplets_unified.csv"

# Determine all possible keys (ensure consistent column order)
fieldnames = ["head", "relation", "tail", "source", "pubmed_ids", "chemical_id", "disease_id"]

# Write to CSV
print(f"Saving unified triplets to {csv_output_path} ...")
with open(csv_output_path, "w", newline='', encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for triplet in unique_triplets:
        row = {key: triplet.get(key, "") for key in fieldnames}
        # Convert list of pubmed_ids to pipe-separated string (if any)
        if isinstance(row["pubmed_ids"], list):
            row["pubmed_ids"] = "|".join(map(str, row["pubmed_ids"]))
        writer.writerow(row)

print("CSV save complete.")


Saving unified triplets to triplets_unified.csv ...
CSV save complete.
